# Home Health Visit-Note Feature Extraction — Hugging Face Edition

A drop-in replacement for the Gemini notebook using **open-source Hugging Face models**.
Two architectural paths, both demonstrated here:

| Path | Primary model | When |
|---|---|---|
| **A. NER + rules** (default) | `Ihor/gliner-biomed-large-v1.0` (~470M params) | High throughput, deterministic, CPU-friendly, audit-grade evidence quotes "for free" (character offsets) |
| **B. Local clinical LLM** | `google/medgemma-4b-it` (multimodal med-Gemma, instruction-tuned) | When you need reasoning over the whole note — overall status, escalation rationale, free-text gap detection |

Both run **on-prem / in-VPC**, which is why the CenterWell roadmap names them as the
PHI-safe alternatives to cloud Gemini for sensitive flows.

This notebook walks Path A end-to-end (executable on CPU), then shows the Path B
cell so you can drop in MedGemma when you have a GPU.

**Other HF models we'll reference (one-liner each):**
- `nvidia/gliner-PII` (Oct 2025) — PHI/PII redaction pass; sits in front of either path when notes leave the BAA enclave.
- `emilyalsentzer/Bio_ClinicalBERT` — embeddings for the note vector index (`visit_note_embeddings_gold`); cited by name in the CenterWell tech stack.
- `Clinical-AI-Apollo/Medical-NER` and `blaze999/Medical-NER` — fixed-label clinical NER alternatives (41 entity types); use these if you'd rather a fine-tuned classifier than a zero-shot generalist.
- `aaditya/Llama3-OpenBioLLM-8B` / `epfl-llm/meditron-7b` — alternative local clinical LLMs if MedGemma's license terms don't fit.

> ⚠️ Sample notes are **synthetic**. Never run a non-BAA cloud model on real PHI;
> the whole point of this stack is that it can run inside your enclave.


## 1. Install dependencies

In [ ]:
# Run once. Comment out after first execution.
%pip install -q "gliner>=0.2.13" "transformers>=4.45" "torch>=2.2" pydantic pandas accelerate

## 2. The feature schema (same as the Gemini notebook)

Identical Pydantic model so you can A/B the two pipelines on the same shape of output.


In [ ]:
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field


class Dyspnea(str, Enum):
    NONE = "none"
    ON_EXERTION = "on_exertion"
    AT_REST = "at_rest"
    NOT_DOCUMENTED = "not_documented"


class Status(str, Enum):
    STABLE = "stable"
    IMPROVING = "improving"
    DECLINING = "declining"
    ACUTE_CONCERN = "acute_concern"


class DxCluster(str, Enum):
    HF = "HF"
    COPD = "COPD"
    DM = "DM"
    POST_SURGICAL = "post_surgical"
    WOUND = "wound"
    OTHER = "other"


class Vitals(BaseModel):
    bp_systolic: Optional[int] = None
    bp_diastolic: Optional[int] = None
    heart_rate: Optional[int] = None
    spo2_percent: Optional[int] = None
    weight_lbs: Optional[float] = None
    weight_change_lbs: Optional[float] = None


class Signal(BaseModel):
    name: str
    value: str
    evidence_quote: str
    confidence: float = 1.0  # GLiNER returns a real score; we keep it.


class VisitNoteFeatures(BaseModel):
    primary_dx_cluster: DxCluster
    overall_status: Status
    vitals: Vitals
    dyspnea: Dyspnea
    dyspnea_evidence: Optional[str] = None
    new_confusion: bool = False
    confusion_evidence: Optional[str] = None
    fall_since_last_visit: bool = False
    prn_diuretic_used: bool = False
    adherence_concern: bool = False
    caregiver_present: bool = False
    escalation_indicators: list[str] = Field(default_factory=list)
    flat_signals: list[Signal] = Field(default_factory=list)
    rationale: str = ""


print("Schema OK")

## 3. Sample synthetic notes (same three)

In [ ]:
NOTES = {
    "note_01_hfref": """
Patient: R.T. (synthetic). Visit type: Routine SN follow-up, episode day 14.
Primary dx: HFrEF (LVEF 28%), T2DM, CKD stage 3a.

Subjective: Pt reports increased SOB with ambulation to bathroom over past 3 days.
Denies chest pain. Daughter (primary caregiver, present at visit) states pt
"seems more confused than usual" for the past 2 days. Pt acknowledges taking
PRN furosemide 40 mg last night for "puffiness."

Objective: BP 158/92 (last visit 142/86). HR 96, RR 22, SpO2 93% on RA.
Weight 187 lbs - up 5.3 lbs from 7 days ago. Lungs: faint crackles bilateral
bases. JVD 1+ at 30 degrees. 2+ pitting edema bilateral lower extremities.

Plan: Notify HF case manager and PCP today. Increase visit frequency to 3x/week.
""",

    "note_02_copd_stable": """
Patient: J.W. (synthetic). Visit type: Routine SN follow-up, week 4.
Primary dx: COPD GOLD III. Lives with adult son (caregiver at work today).

Subjective: Pt stable per self-report. Reports continued use of albuterol rescue
inhaler "a couple times a day," consistent with baseline. Denies new sputum,
fever, or worsening cough.

Objective: BP 138/82, HR 88, RR 18, SpO2 92% on RA. Weight stable at 162 lbs.
Lungs: diminished breath sounds bilaterally with mild prolonged expiration.
No wheezes. Inhaler technique adequate after a quick reminder.

Plan: Continue current regimen. Next visit in 4 days.
""",

    "note_03_dm_sdoh": """
Patient: E.B. (synthetic). Visit type: Routine SN follow-up, week 6.
Primary dx: T2DM with peripheral neuropathy, A1c 9.4%. Lives alone; recently
widowed. No caregiver at visit.

Subjective: Glucose readings 78 to 312 mg/dL last week per log. Admits to
skipping evening insulin twice last week - "I ran out and didn't want to
bother my daughter for a ride." Endorses low mood, poor appetite, "some
nights I just don't eat - the fridge is mostly empty." Denies SI.

Objective: BP 152/88, HR 82, SpO2 98% on RA. Weight 218 lbs - down 4 lbs
(unintentional). Fingerstick at visit: 248 mg/dL. Insulin pen empty since
last Friday. PHQ-2 positive (score 5).

Plan: Notify PCP today - urgent insulin refill. Refer to social worker for
food insecurity and transportation. Increase visit frequency to 2x/week.
""",
}

for k, v in NOTES.items():
    print(f"{k}: {len(v)} chars")

## 4. Load GLiNER-BioMed

`Ihor/gliner-biomed-large-v1.0` (DS4DH × University of Geneva, 2025) is purpose-built
for biomedical zero-shot NER. It takes **arbitrary entity labels at inference time** —
which means we can define labels that map directly onto our Pydantic schema fields
instead of relying on a fixed taxonomy.

The model is ~470M params; CPU works but a GPU helps. First run downloads ~1.8 GB.


In [ ]:
from gliner import GLiNER

# Primary recommendation. Alternatives, in case you need them:
#   - "Ihor/gliner-biomed-small-v1.0"  (~165M, faster, slightly lower F1)
#   - "Ihor/gliner-biomed-base-v1.0"   (~210M, mid)
#   - "gliner-community/gliner_medium-v2.5" (general-purpose, not biomed-specific)
MODEL_NAME = "Ihor/gliner-biomed-large-v1.0"

ner_model = GLiNER.from_pretrained(MODEL_NAME)
print(f"Loaded {MODEL_NAME}")

## 5. Define entity labels that mirror the schema

This is the core idea of zero-shot NER: you choose the labels. We pick labels that
either directly fill a schema field (e.g. `"blood pressure measurement"`) or
clearly map to one via a small rule (e.g. `"new-onset confusion"` → `new_confusion=True`).

These labels were tuned against the three sample notes; for a real deployment
you'd refine them on a held-out set of ~50-100 notes.


In [ ]:
ENTITY_LABELS = [
    # Vitals
    "blood pressure measurement",
    "heart rate measurement",
    "respiratory rate measurement",
    "oxygen saturation measurement",
    "body weight",
    "weight change",
    "temperature measurement",
    "blood glucose measurement",

    # Cardiopulmonary symptoms
    "dyspnea on exertion",
    "dyspnea at rest",
    "shortness of breath",
    "crackles",
    "wheezes",
    "jugular venous distension",
    "peripheral edema",

    # Neurocognitive
    "new-onset confusion",
    "altered mental status",

    # Medication / adherence
    "diuretic medication use",
    "rescue inhaler use",
    "medication non-adherence",
    "missed medication dose",
    "insulin use",

    # Safety
    "fall event",
    "home safety hazard",

    # Social / context
    "caregiver presence at visit",
    "food insecurity",
    "social isolation",

    # Diagnoses
    "heart failure",
    "COPD diagnosis",
    "diabetes diagnosis",
    "wound assessment",
    "post-surgical status",
]

print(f"{len(ENTITY_LABELS)} entity labels defined")

## 6. Run NER on one note (inspection)

Let's see what GLiNER pulls out of the HFrEF note. Each result has a character
span, label, and confidence score — perfect substrate for the assembly logic
below.


In [ ]:
note_text = NOTES["note_01_hfref"]
entities = ner_model.predict_entities(note_text, ENTITY_LABELS, threshold=0.4)

print(f"Found {len(entities)} entities\n")
print(f"{'label':<35} {'text':<40} score")
print("-" * 90)
for e in entities:
    print(f"{e['label']:<35} {e['text'][:38]:<40} {e['score']:.3f}")

## 7. Assembly: NER spans + regex → Pydantic schema

GLiNER tells us **what** entities exist and **where** in the note. The assembly
step turns those into schema fields:

- Numeric vitals: regex (cheaper, more reliable than NER for "158/92").
- Symptom flags: presence of an entity of the right label sets the field, with
  the entity text as the evidence quote.
- Overall status, escalation indicators: small rule that fires based on which
  combinations of entities show up. (In production, this is a Bio_ClinicalBERT
  classifier fine-tuned on labelled visits — or you delegate to MedGemma in
  Section 10 below.)


In [ ]:

import re
from typing import Optional


# --- helpers ---------------------------------------------------------------

def _entities_by_label(entities: list[dict]) -> dict[str, list[dict]]:
    """Group GLiNER entities by their label."""
    out: dict[str, list[dict]] = {}
    for e in entities:
        out.setdefault(e["label"], []).append(e)
    return out


def _first_text(grouped: dict[str, list[dict]], label: str) -> Optional[str]:
    """Verbatim text of the first occurrence of label, if any."""
    items = grouped.get(label) or []
    return items[0]["text"] if items else None


def _has(grouped: dict, label: str) -> bool:
    return bool(grouped.get(label))


# --- regex-based extractors for vitals (fast, precise) ---------------------

def parse_vitals(text: str) -> Vitals:
    v = Vitals()

    m = re.search(r"BP\s*(\d{2,3})\s*/\s*(\d{2,3})", text)
    if m:
        v.bp_systolic, v.bp_diastolic = int(m.group(1)), int(m.group(2))

    m = re.search(r"\bHR\s*(\d{2,3})\b", text)
    if m:
        v.heart_rate = int(m.group(1))

    m = re.search(r"SpO2\s*(\d{2,3})\s*%", text, re.IGNORECASE)
    if m:
        v.spo2_percent = int(m.group(1))

    m = re.search(r"[Ww]eight\s+(\d{2,3}(?:\.\d+)?)\s*lbs", text)
    if m:
        v.weight_lbs = float(m.group(1))

    # Weight delta: matches "up 5.3 lbs", "down 4 lbs"
    m = re.search(r"\b(up|down)\s+(\d+(?:\.\d+)?)\s*lbs", text, re.IGNORECASE)
    if m:
        sign = 1.0 if m.group(1).lower() == "up" else -1.0
        v.weight_change_lbs = sign * float(m.group(2))

    return v


# --- dyspnea level: NER labels are graded, so pick the strongest -----------

def parse_dyspnea(grouped: dict) -> tuple[Dyspnea, Optional[str]]:
    if _has(grouped, "dyspnea at rest"):
        return Dyspnea.AT_REST, _first_text(grouped, "dyspnea at rest")
    if _has(grouped, "dyspnea on exertion"):
        return Dyspnea.ON_EXERTION, _first_text(grouped, "dyspnea on exertion")
    # Fall back to generic "shortness of breath" if no severity label fired
    sob = _first_text(grouped, "shortness of breath")
    if sob:
        # If we caught SOB but no severity tag, treat it as unspecified-exertional
        # for HF/COPD notes; for cleaner data, you'd retrain with finer labels.
        return Dyspnea.ON_EXERTION, sob
    return Dyspnea.NOT_DOCUMENTED, None


# --- dx cluster: simple priority by which dx-entity fires ------------------

DX_PRIORITY = [
    ("heart failure", DxCluster.HF),
    ("COPD diagnosis", DxCluster.COPD),
    ("diabetes diagnosis", DxCluster.DM),
    ("wound assessment", DxCluster.WOUND),
    ("post-surgical status", DxCluster.POST_SURGICAL),
]


def parse_dx_cluster(grouped: dict) -> DxCluster:
    for label, cluster in DX_PRIORITY:
        if _has(grouped, label):
            return cluster
    return DxCluster.OTHER


# --- overall status: rule on combinations ----------------------------------
# In production this is replaced by a Bio_ClinicalBERT classifier or MedGemma.

def parse_status(grouped: dict, vitals: Vitals) -> tuple[Status, list[str]]:
    escalations: list[str] = []

    # Hard escalations
    if vitals.weight_change_lbs is not None and vitals.weight_change_lbs >= 3.0:
        escalations.append(
            f"rapid weight gain {vitals.weight_change_lbs:+.1f} lbs"
        )
    if _has(grouped, "new-onset confusion"):
        escalations.append("new-onset confusion documented")
    if _has(grouped, "jugular venous distension") and _has(grouped, "crackles"):
        escalations.append("JVD with crackles (volume overload pattern)")
    if _has(grouped, "fall event"):
        escalations.append("fall since last visit")
    if _has(grouped, "missed medication dose") or _has(grouped, "medication non-adherence"):
        escalations.append("medication adherence gap")
    if _has(grouped, "food insecurity"):
        escalations.append("food insecurity reported")

    if escalations:
        return Status.DECLINING, escalations
    return Status.STABLE, escalations


# --- main assembler --------------------------------------------------------

def assemble_features(note_text: str, entities: list[dict]) -> VisitNoteFeatures:
    grouped = _entities_by_label(entities)
    vitals = parse_vitals(note_text)
    dyspnea, dyspnea_evidence = parse_dyspnea(grouped)
    dx_cluster = parse_dx_cluster(grouped)
    status, escalations = parse_status(grouped, vitals)

    # Build flat_signals from the strongest NER hits, carrying their real scores.
    SIGNAL_LABELS = {
        "dyspnea on exertion", "dyspnea at rest", "shortness of breath",
        "new-onset confusion", "jugular venous distension", "peripheral edema",
        "crackles", "diuretic medication use", "rescue inhaler use",
        "fall event", "food insecurity", "social isolation",
        "missed medication dose", "medication non-adherence",
    }
    flat = []
    for label, items in grouped.items():
        if label not in SIGNAL_LABELS:
            continue
        best = max(items, key=lambda e: e["score"])
        flat.append(Signal(
            name=label.replace(" ", "_"),
            value="present",
            evidence_quote=best["text"],
            confidence=round(float(best["score"]), 3),
        ))

    return VisitNoteFeatures(
        primary_dx_cluster=dx_cluster,
        overall_status=status,
        vitals=vitals,
        dyspnea=dyspnea,
        dyspnea_evidence=dyspnea_evidence,
        new_confusion=_has(grouped, "new-onset confusion"),
        confusion_evidence=_first_text(grouped, "new-onset confusion"),
        fall_since_last_visit=_has(grouped, "fall event"),
        prn_diuretic_used=_has(grouped, "diuretic medication use"),
        adherence_concern=(_has(grouped, "medication non-adherence")
                           or _has(grouped, "missed medication dose")),
        caregiver_present=_has(grouped, "caregiver presence at visit"),
        escalation_indicators=escalations,
        flat_signals=flat,
        rationale=(
            f"NER+rules pipeline. dx={dx_cluster.value}, "
            f"{len(escalations)} escalation indicator(s), {len(flat)} signal(s)."
        ),
    )


## 8. Extract features for all three notes


In [ ]:
def extract_features(note_text: str) -> VisitNoteFeatures:
    entities = ner_model.predict_entities(note_text, ENTITY_LABELS, threshold=0.4)
    return assemble_features(note_text, entities)


results = {}
for note_id, text in NOTES.items():
    print(f"Extracting {note_id}...")
    results[note_id] = extract_features(text)
print(f"\nExtracted features for {len(results)} notes")

## 9. DataFrame view

In [ ]:
import pandas as pd

rows = []
for note_id, f in results.items():
    rows.append({
        "note_id": note_id,
        "dx_cluster": f.primary_dx_cluster.value,
        "status": f.overall_status.value,
        "dyspnea": f.dyspnea.value,
        "BP": f"{f.vitals.bp_systolic}/{f.vitals.bp_diastolic}" if f.vitals.bp_systolic else None,
        "spo2": f.vitals.spo2_percent,
        "wt_delta": f.vitals.weight_change_lbs,
        "new_confusion": f.new_confusion,
        "fall": f.fall_since_last_visit,
        "prn_diuretic": f.prn_diuretic_used,
        "adherence_concern": f.adherence_concern,
        "n_escalations": len(f.escalation_indicators),
    })

pd.DataFrame(rows).set_index("note_id")

## 10. Audit trail — each signal traces back to a verbatim span

In [ ]:
note_id = "note_01_hfref"
print(f"=== {note_id} ===\n")
for sig in results[note_id].flat_signals:
    print(f"• {sig.name}  (confidence={sig.confidence})")
    print(f"    evidence: \"{sig.evidence_quote}\"\n")

print("Escalation indicators:")
for e in results[note_id].escalation_indicators:
    print(f"  - {e}")

## 11. (Optional) Path B — MedGemma as a local Gemini replacement

If you need the full reasoning surface that Gemini gave you in the original
notebook (overall status, escalation rationale, free-text gap detection), the
right HF swap is **MedGemma** — Google's medically-tuned Gemma, released
under the Health AI Developer Foundations program. Same instruction-following
pattern as Gemini, but you run it inside your own VPC.

The CenterWell roadmap explicitly names MedGemma 1.5 (4B / 27B) as the on-prem
clinical-reasoning model. The 4B variant fits on a single 24 GB GPU.

This cell is left **un-executed** by default — it needs a GPU and a HF token
with terms-of-use accepted on the MedGemma model card.


In [ ]:
# Requires: a CUDA GPU + `huggingface-cli login` + license acceptance on
# https://huggingface.co/google/medgemma-4b-it

# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer
#
# MEDGEMMA_MODEL = "google/medgemma-4b-it"
# tok = AutoTokenizer.from_pretrained(MEDGEMMA_MODEL)
# llm = AutoModelForCausalLM.from_pretrained(
#     MEDGEMMA_MODEL,
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
# )
#
# import json
# SCHEMA_HINT = json.dumps(VisitNoteFeatures.model_json_schema(), indent=2)
#
# def medgemma_extract(note_text: str) -> VisitNoteFeatures:
#     prompt = f"""You are a clinical NLP extractor for a Home Health agency.
# Read the visit note below and emit ONLY a JSON object matching the schema.
# Every clinical finding must carry a verbatim evidence quote from the note.
#
# Schema (Pydantic JSON schema):
# {SCHEMA_HINT}
#
# Visit note:
# \"\"\"
# {note_text.strip()}
# \"\"\"
#
# JSON output:"""
#
#     messages = [{"role": "user", "content": prompt}]
#     inputs = tok.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(llm.device)
#     out = llm.generate(inputs, max_new_tokens=2048, do_sample=False, temperature=0.0)
#     text = tok.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
#     # Strip code fences if MedGemma wraps the JSON
#     text = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
#     return VisitNoteFeatures.model_validate_json(text)
#
# medgemma_extract(NOTES["note_01_hfref"])

print("MedGemma cell is template-only. Uncomment and run on a GPU host with license accepted.")

## 12. Which approach when?

| | Gemini 2.5 Flash | GLiNER-BioMed (this notebook) | MedGemma 4B |
|---|---|---|---|
| Where it runs | Vertex AI (BAA) | Anywhere — CPU OK, GPU faster | Your GPU, your VPC |
| Cost per note | ~$0.0001 | $0 marginal | $0 marginal, GPU sunk cost |
| Reasoning depth | Full (rationale, gap detection, multi-hop) | Entity-level only; rules for the rest | Full (LLM) |
| Evidence quotes | Prompt-driven (model has to comply) | Free — they're character spans | Prompt-driven |
| Cold-start cost | $0 | Tune label set on ~50-100 notes | License acceptance + GPU |
| Determinism | Low (T=0 helps, not perfect) | High | Low |
| Best for | Hard, narrative-heavy reasoning; small volume | High-volume structured extraction; audit-heavy workflows | Same as Gemini but PHI must stay in-VPC |

**Pragmatic recommendation for the CenterWell platform** — matches what the
roadmap implies:

- **Real-time feature extraction (visit save → feature store):** GLiNER-BioMed + rules. Cheap, fast, deterministic.
- **Visit-note QA / completeness checks:** Gemini 2.5 Flash (cloud, BAA) or MedGemma 4B (on-prem). Needs reasoning.
- **PHI scrub before any external egress:** `nvidia/gliner-PII` or `urchade/gliner_multi_pii-v1` (roadmap explicitly names the latter).
- **Note embeddings for the vector index:** Bio_ClinicalBERT (roadmap-named).
- **Concept linking to UMLS/SNOMED:** SapBERT (roadmap-named).

Stacking these gives you a fully open-source pipeline that fits inside the
BAA enclave end-to-end — useful when you need to demonstrate to compliance
that no clinical text ever leaves the perimeter.
